# EfficientNet FE + XGB — Paper Revision

EfficientNetB0 frozen (ImageNet weights) used as feature extractor. No data leakage: the feature extractor is fixed and does not learn from the dataset. XGBoost trained per fold on extracted features.

Added for revision: animal-level metrics, temporal analysis, bootstrap CIs, per-fold prediction saving.

## Method
1. Load full dataset from TFRecord (`full_ds_fixed_no_control.tfrecord`)
2. Extract 256-dim features from frozen EfficientNetB0 (ImageNet weights, all layers frozen) — **not** data leakage since the extractor is fixed
3. Leave-One-Group-Out (LOGO) CV: 10 folds (one per mouse)
4. Per fold: oversample minority class in train → XGBoost train → predict on held-out mouse
5. Pool predictions and compute exam-level, animal-level, temporal, bootstrap CI metrics

In [ ]:
# NOTE: Requires GPU (TF 2.10 + CUDA 11.x). Cannot run in CPU-only venv.
import os
import sys

# Set working directory to repo root so config and src are importable.
# The kernel may start from a different directory when run via nbconvert.
_candidate = os.path.abspath('.')
if not os.path.isfile(os.path.join(_candidate, 'config.py')):
    _search = _candidate
    for _ in range(5):
        _search = os.path.dirname(_search)
        if os.path.isfile(os.path.join(_search, 'config.py')):
            _candidate = _search
            break
os.chdir(_candidate)
sys.path.insert(0, _candidate)

print('Working directory:', os.getcwd())

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
print(tf.__version__)
print(tf.test.is_gpu_available())

In [ ]:
import warnings
import random
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import gc

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

import config
from tfrecordhandlerDL import TFRecordDataHandlerDL

warnings.filterwarnings('ignore')

print('XGBoost version:', xgb.__version__)
print('TF version:', tf.__version__)

## Configuration

Output directory for paper revision DL feature extraction results.

In [ ]:
# Output directory for this experiment (paper revision)
OUTPUT_DIR = config.OUTPUT_EXP_DL_FE
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
INPUT_DIR = 'input/deep_learning'
BATCH_SIZE = 32
SEED = 42
N_BOOTSTRAP = 10_000

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print('Output directory:', OUTPUT_DIR)

## Data Loading

Load the full dataset from TFRecord (no control mice). The TFRecord handler returns batches of
`(images, masks, labels, mouse_ids, days_of_study)`.

In [ ]:
def visualize_data(image_batch, mask_batch, num_samples=4):
    plt.figure(figsize=(10, num_samples * 2))
    for i in range(num_samples):
        plt.subplot(num_samples, 2, 2 * i + 1)
        plt.imshow(np.squeeze(image_batch[i]), cmap='gray')
        plt.title(f'Image {i}')
        plt.axis('off')
        plt.subplot(num_samples, 2, 2 * i + 2)
        plt.imshow(np.squeeze(mask_batch[i]), cmap='gray')
        plt.title(f'Mask {i}')
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
train_tfrecord = os.path.join(INPUT_DIR, 'full_ds_fixed_no_control.tfrecord')

train_ds_handler = TFRecordDataHandlerDL(train_tfrecord, batch_size=BATCH_SIZE, shuffle=False, augment=False)
train_ds = train_ds_handler.dataset
print(f'Number of samples: {train_ds_handler.length}')
print(f'Number of batches: {train_ds_handler.length // BATCH_SIZE}')

images_tf, masks_tf, group_names, mouse_ids, days_of_study = next(iter(train_ds))
print(f'Images min: {images_tf.numpy().min():.4f}, max: {images_tf.numpy().max():.4f}')

visualize_data(images_tf, masks_tf, num_samples=4)

## Augmentation and Balancing Helpers

Defined for reference — augmentation is NOT used during frozen-FE extraction (it would add
stochasticity without benefit since we are not fine-tuning). Oversampling is done per fold in
numpy after feature extraction.

In [ ]:
def augment_data(image, mask, label, mouse_ids, days_of_study):
    seed = tf.random.uniform([], maxval=10000, dtype=tf.int32)
    image = tf.image.stateless_random_flip_left_right(image, seed=[seed, 1])
    mask = tf.image.stateless_random_flip_left_right(mask, seed=[seed, 1])
    image = tf.image.stateless_random_brightness(image, max_delta=0.2, seed=[seed, 2])
    image = tf.image.stateless_random_contrast(image, lower=0.8, upper=1.2, seed=[seed, 3])
    image = tf.image.stateless_random_saturation(image, lower=0.8, upper=1.2, seed=[seed, 4])
    image = tf.image.stateless_random_hue(image, max_delta=0.05, seed=[seed, 5])
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, mask, label, mouse_ids, days_of_study

## Feature Extractor — Frozen EfficientNetB0

All EfficientNetB0 layers are frozen (`trainable=False`). The network uses ImageNet weights and is
never updated during this pipeline. Features are extracted from the penultimate Dense(256) layer.

**No data leakage:** the extractor is fixed — no information from any animal flows into the weights.
This is analogous to using a fixed handcrafted feature transform.

In [ ]:
# Build frozen EfficientNetB0 feature extractor
input_layer = tf.keras.layers.Input(shape=(256, 256, 3), name='input_layer')
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=input_layer)
for layer in base_model.layers:
    layer.trainable = False  # ALL layers frozen — no learning from dataset

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
output_layer = layers.Dense(1, activation='sigmoid')(x)

# Full model (not used for training — only for architecture reference)
model = tf.keras.models.Model(inputs=input_layer, outputs=output_layer)

# Feature extractor: outputs the 256-dim Dense layer (before final sigmoid)
feature_extractor = models.Model(inputs=input_layer, outputs=x)

print('Available devices:')
print(tf.config.list_physical_devices('GPU'))
print(f'Feature extractor output shape: {feature_extractor.output_shape}')
print(f'Trainable parameters: {sum(v.numpy().size for v in feature_extractor.trainable_variables)}')
print(f'Non-trainable (frozen) parameters: {sum(v.numpy().size for v in feature_extractor.non_trainable_variables)}')

## Extract Features from Full Dataset

Features are extracted from ALL samples at once (before any fold split). This is correct and not
leakage because the extractor is frozen — the test mouse's images cannot influence the extractor
weights, which remain at ImageNet values throughout.

In [ ]:
def extract_features_labels_from_dataset(dataset, allowed_days, feature_extractor):
    """
    Extract 256-dim features from frozen EfficientNetB0, retaining mouse_id and day_of_study.

    Args:
        dataset: tf.data.Dataset yielding (images, masks, labels, mouse_ids, days)
        allowed_days: numpy array of days to include (filter)
        feature_extractor: frozen Keras model

    Returns:
        X (np.ndarray): features (N, 256)
        y (np.ndarray): labels (N,)
        mouse_ids (np.ndarray): string mouse IDs (N,)
        days (np.ndarray): day_of_study integers (N,)
    """
    start_time = time.time()
    print('[INFO] Extracting features from full dataset ...')

    x_batches, labels_all, mouse_ids_all, days_all = [], [], [], []

    for batch in dataset:
        x_batch, masks_b, y_batch, m_ids, days_b = batch
        days_np = days_b.numpy()

        mask_days = np.isin(days_np, allowed_days)
        if not np.any(mask_days):
            continue
        mask_days_tf = tf.reshape(mask_days, [-1])

        x_batch  = tf.boolean_mask(x_batch, mask_days_tf)
        y_batch  = tf.boolean_mask(y_batch, mask_days_tf)
        m_ids    = tf.boolean_mask(m_ids, mask_days_tf)
        days_b   = tf.boolean_mask(days_b, mask_days_tf)

        x_batches.append(x_batch)
        labels_all.append(y_batch.numpy())
        mouse_ids_all.append(m_ids.numpy())
        days_all.append(days_b.numpy())

    if not x_batches:
        return None, None, None, None

    x_all = tf.concat(x_batches, axis=0)

    print(f'[INFO] Total samples: {x_all.shape[0]} — running feature_extractor.predict ...')
    X = feature_extractor.predict(x_all, verbose=1, batch_size=32)

    y          = np.concatenate(labels_all, axis=0)
    mouse_ids  = np.concatenate(mouse_ids_all, axis=0)
    days       = np.concatenate(days_all, axis=0)

    print(f'[INFO] Extraction complete in {time.time() - start_time:.1f}s')
    return X, y, mouse_ids, days


# Re-batch at standard size
train_ds_loop = train_ds.unbatch().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Collect all days
all_days_list = []
for batch in train_ds_loop:
    _, _, _, _, days_b = batch
    all_days_list.extend(days_b.numpy())
all_days_list = np.unique(np.array(all_days_list))

# Extract features
all_features, all_labels, all_mouse_ids, all_days_array = extract_features_labels_from_dataset(
    train_ds_loop, allowed_days=all_days_list, feature_extractor=feature_extractor
)

if all_features is None:
    raise ValueError('Feature extraction failed — no samples returned.')

# Fix shapes
all_labels       = np.squeeze(all_labels)
all_mouse_ids    = np.squeeze(all_mouse_ids)
all_days_array   = np.squeeze(all_days_array)

# Decode bytes to str if needed
if isinstance(all_mouse_ids[0], bytes):
    all_mouse_ids = np.array([m.decode('utf-8') for m in all_mouse_ids])

print(f'\nFeatures shape: {all_features.shape}')
print(f'Labels shape:   {all_labels.shape}')
print(f'Mouse IDs:      {np.unique(all_mouse_ids)}')
print(f'Days range:     {all_days_array.min()} – {all_days_array.max()}')
print(f'Label counts:   0={int((all_labels==0).sum())}  1={int((all_labels==1).sum())}')

## LOGO Cross-Validation — XGBoost on Extracted Features

For each fold (one mouse left out as test):
1. Oversample minority class in **training data only** (numpy random choice)
2. Train XGBoost on balanced training features
3. Predict on held-out mouse exams
4. Track mouse_id and day_of_study per prediction
5. Save per-fold `predictions.csv`

In [ ]:
logo = LeaveOneGroupOut()

# Pooled accumulators across all folds
all_y_true         = []
all_y_pred         = []
all_y_proba        = []
all_mouse_ids_pred = []   # mouse IDs for each test prediction
all_days_pred      = []   # days for each test prediction

# Feature importance accumulator (keyed by feature index)
feature_importance_accum = defaultdict(list)
n_features = all_features.shape[1]
feature_names = [f'feat_{i}' for i in range(n_features)]

print(f'Running LOGO CV ({logo.get_n_splits(groups=all_mouse_ids)} folds) ...')
print('=' * 60)

for fold, (train_idx, test_idx) in enumerate(logo.split(all_features, all_labels, groups=all_mouse_ids)):
    test_mouse = np.unique(all_mouse_ids[test_idx])
    print(f'\nFold {fold+1}: test mouse = {test_mouse}, n_test_exams = {len(test_idx)}')

    X_train, y_train = all_features[train_idx], all_labels[train_idx]
    X_test,  y_test  = all_features[test_idx],  all_labels[test_idx]

    # --- Oversample minority class in TRAIN only ---
    pos_idx = np.where(y_train == 1)[0]
    neg_idx = np.where(y_train == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f'  Fold {fold+1} skipped — missing class in training set.')
        continue

    if len(pos_idx) < len(neg_idx):
        pos_oversampled = np.random.choice(pos_idx, size=len(neg_idx), replace=True)
        X_train_bal = np.concatenate([X_train[neg_idx], X_train[pos_oversampled]], axis=0)
        y_train_bal = np.concatenate([y_train[neg_idx], y_train[pos_oversampled]], axis=0)
    else:
        neg_oversampled = np.random.choice(neg_idx, size=len(pos_idx), replace=True)
        X_train_bal = np.concatenate([X_train[pos_idx], X_train[neg_oversampled]], axis=0)
        y_train_bal = np.concatenate([y_train[pos_idx], y_train[neg_oversampled]], axis=0)

    # Shuffle
    shuf = np.random.permutation(len(y_train_bal))
    X_train_bal = X_train_bal[shuf]
    y_train_bal = y_train_bal[shuf]

    print(f'  X_train_balanced: {X_train_bal.shape} | X_test: {X_test.shape}')

    # --- Train XGBoost (no use_label_encoder — deprecated in newer xgboost) ---
    xgb_clf = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.01,
        max_depth=3,
        subsample=0.6,
        colsample_bytree=0.6,
        eval_metric='logloss',
        random_state=SEED,
    )
    xgb_clf.fit(X_train_bal, y_train_bal)

    # --- Predict ---
    y_proba = xgb_clf.predict_proba(X_test)[:, 1]
    y_pred  = xgb_clf.predict(X_test)

    # --- Feature importances ---
    for feat, imp in zip(feature_names, xgb_clf.feature_importances_):
        feature_importance_accum[feat].append(float(imp))

    # --- Save per-fold predictions CSV ---
    fold_dir = os.path.join(OUTPUT_DIR, f'fold_{fold+1}')
    os.makedirs(fold_dir, exist_ok=True)
    pd.DataFrame({
        'm_id':        all_mouse_ids[test_idx],
        'day_of_study': all_days_array[test_idx],
        'y_true':      y_test,
        'y_pred':      y_pred,
        'y_proba':     y_proba,
    }).to_csv(os.path.join(fold_dir, 'predictions.csv'), index=False)

    # --- Accumulate ---
    all_y_true.extend(y_test.tolist())
    all_y_pred.extend(y_pred.tolist())
    all_y_proba.extend(y_proba.tolist())
    all_mouse_ids_pred.extend(all_mouse_ids[test_idx].tolist())
    all_days_pred.extend(all_days_array[test_idx].tolist())

print('\n' + '=' * 60)
print(f'All folds complete. Pooled exams: {len(all_y_true)}')

# Convert to arrays
all_y_true         = np.array(all_y_true)
all_y_pred         = np.array(all_y_pred)
all_y_proba        = np.array(all_y_proba)
all_mouse_ids_pred = np.array(all_mouse_ids_pred)
all_days_pred      = np.array(all_days_pred)

## A1 — Exam-Level Results

Pool all fold predictions and compute metrics at the exam level (each MRI scan is one observation).

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score

def compute_metrics(y_true, y_pred, y_proba):
    """
    Compute classification metrics from pooled predictions.
    Uses labels=[0,1] in confusion_matrix to ensure 2x2 matrix even
    when one class is absent from predictions.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    spec = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    npv  = tn / (tn + fn) if (tn + fn) > 0 else float('nan')
    acc  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else float('nan')

    try:
        auc_val = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc_val = float('nan')

    return {
        'AUC': auc_val,
        'Sensitivity': sens,
        'Specificity': spec,
        'PPV': ppv,
        'NPV': npv,
        'Accuracy': acc,
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
    }


exam_metrics = compute_metrics(all_y_true, all_y_pred, all_y_proba)

print('Exam-Level Metrics (pooled across all LOGO folds):')
print('-' * 45)
for k, v in exam_metrics.items():
    if isinstance(v, float):
        print(f'  {k:<14}: {v:.4f}')
    else:
        print(f'  {k:<14}: {v}')

# Save
exam_out = os.path.join(OUTPUT_DIR, 'summary_metrics_exam_level.csv')
pd.DataFrame([exam_metrics]).to_csv(exam_out, index=False)
print(f'\nSaved: {exam_out}')

## A2 — Animal-Level Results (Majority Vote)

Aggregate exam-level predictions to the mouse level using **majority vote** on `y_pred` across
all scans for that mouse. Probability is averaged. This addresses Reviewer Comment A2.

In [ ]:
# Build exam-level results DataFrame
results_df = pd.DataFrame({
    'mouse_id': all_mouse_ids_pred,
    'y_true':   all_y_true,
    'y_pred':   all_y_pred,
    'y_proba':  all_y_proba,
})

# Majority vote per mouse (A2)
animal_df = results_df.groupby('mouse_id').agg(
    y_true=('y_true', 'first'),
    y_pred_vote=('y_pred', lambda x: int(x.mode()[0])),
    y_proba_mean=('y_proba', 'mean'),
).reset_index()

print('Animal-Level Predictions (majority vote across exams):')
print(animal_df.to_string(index=False))
print()

# Animal-level metrics
if len(animal_df['y_true'].unique()) > 1:
    animal_metrics = compute_metrics(
        animal_df['y_true'].values,
        animal_df['y_pred_vote'].values,
        animal_df['y_proba_mean'].values,
    )
else:
    animal_metrics = {k: float('nan') for k in
                      ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Accuracy',
                       'TP', 'TN', 'FP', 'FN']}

print(f'Animal-Level Metrics (N={len(animal_df)} mice):')
print('-' * 40)
for k, v in animal_metrics.items():
    if isinstance(v, float):
        print(f'  {k:<14}: {v:.4f}')
    else:
        print(f'  {k:<14}: {v}')

# Save
animal_df.to_csv(os.path.join(OUTPUT_DIR, 'summary_metrics_animal_level.csv'), index=False)
pd.DataFrame([animal_metrics]).to_csv(os.path.join(OUTPUT_DIR, 'animal_level_metrics.csv'), index=False)
print('\nSaved: summary_metrics_animal_level.csv, animal_level_metrics.csv')

## A2b — Animal-Level Results — Day-Weighted Vote

Alternative to majority vote: each exam's predicted probability is **weighted by its day of study**
before aggregating per animal. Two weighting schemes compared:
- **Linear:** weight = day_of_study (proportional to time)
- **Quadratic:** weight = day_of_study² (stronger late-bias)

In [ ]:
weighted_rows = []
for mouse in np.unique(all_mouse_ids_pred):
    mask   = all_mouse_ids_pred == mouse
    days   = all_days_pred[mask].astype(float)
    proba  = all_y_proba[mask]
    y_true = all_y_true[mask][0]

    w_lin  = days / days.sum()
    prob_lin  = float(np.dot(w_lin, proba))

    w_quad = days**2 / (days**2).sum()
    prob_quad = float(np.dot(w_quad, proba))

    weighted_rows.append({
        'mouse_id':     mouse,
        'y_true':       int(y_true),
        'n_exams':      int(mask.sum()),
        'day_range':    f'{int(days.min())}–{int(days.max())}',
        'prob_uniform': float(proba.mean()),
        'pred_uniform': int(proba.mean() > 0.5),
        'prob_linear':  prob_lin,
        'pred_linear':  int(prob_lin > 0.5),
        'prob_quad':    prob_quad,
        'pred_quad':    int(prob_quad > 0.5),
    })

wv_df = pd.DataFrame(weighted_rows)
print('Per-mouse predictions (uniform vs linear vs quadratic weighting):')
print(wv_df.to_string(index=False))
print()

for scheme, pred_col, prob_col in [
    ('Uniform (majority vote)', 'pred_uniform', 'prob_uniform'),
    ('Linear day-weight',       'pred_linear',  'prob_linear'),
    ('Quadratic day-weight',    'pred_quad',    'prob_quad'),
]:
    m = compute_metrics(wv_df['y_true'].values, wv_df[pred_col].values, wv_df[prob_col].values)
    print(f'{scheme}:')
    print(f"  AUC={m['AUC']:.4f}  Sens={m['Sensitivity']:.4f}  "
          f"Spec={m['Specificity']:.4f}  PPV={m['PPV']:.4f}  "
          f"Acc={m['Accuracy']:.4f}  TP={m['TP']} FP={m['FP']} FN={m['FN']} TN={m['TN']}")
    print()

wv_df.to_csv(os.path.join(OUTPUT_DIR, 'animal_level_weighted_vote.csv'), index=False)
print('Saved: animal_level_weighted_vote.csv')

## A3 — Temporal Analysis

Split all pooled predictions into **early / mid / late** terciles based on `day_of_study`,
using the 33rd and 66th percentiles as thresholds. Report AUC, Sensitivity, Specificity per window.

Addresses Reviewer Comment A3: assess whether performance is consistent across the treatment timeline.

In [ ]:
t33, t66 = np.percentile(all_days_pred, [33, 66])
print(f'Tercile thresholds: early <= {t33:.0f}, mid <= {t66:.0f}, late > {t66:.0f} (days)')
print()

temporal_rows = []
for window_name, mask in [
    ('early', all_days_pred <= t33),
    ('mid',   (all_days_pred > t33) & (all_days_pred <= t66)),
    ('late',  all_days_pred > t66),
]:
    yt  = all_y_true[mask]
    yp  = all_y_pred[mask]
    ypr = all_y_proba[mask]
    n   = int(mask.sum())

    threshold_str = (f'<={t33:.0f}' if window_name == 'early' else
                     (f'<={t66:.0f}' if window_name == 'mid' else f'>{t66:.0f}'))

    if n == 0 or len(np.unique(yt)) < 2:
        row = {'window': window_name, 'n_exams': n, 'day_threshold': threshold_str,
               'AUC': float('nan'), 'Sensitivity': float('nan'), 'Specificity': float('nan')}
    else:
        m = compute_metrics(yt, yp, ypr)
        row = {'window': window_name, 'n_exams': n, 'day_threshold': threshold_str,
               'AUC': m['AUC'], 'Sensitivity': m['Sensitivity'], 'Specificity': m['Specificity']}

    temporal_rows.append(row)

temporal_df = pd.DataFrame(temporal_rows)
print('Temporal Analysis (early / mid / late terciles):')
print(temporal_df.to_string(index=False))

temporal_df.to_csv(os.path.join(OUTPUT_DIR, 'temporal_analysis.csv'), index=False)
print('\nSaved: temporal_analysis.csv')

## A4 — Bootstrap 95% Confidence Intervals

Compute 95% CIs for exam-level AUC, Sensitivity, Specificity, PPV, and Accuracy using
**non-parametric bootstrap** with 10,000 resamples (sampling with replacement from the pooled
exam-level predictions). Bootstrap samples where only one class is present are skipped.

Addresses Reviewer Comment A4: quantify uncertainty around reported point estimates.

In [ ]:
n_exams = len(all_y_true)
boot_metrics = defaultdict(list)

np.random.seed(SEED)
for _ in range(N_BOOTSTRAP):
    idx   = np.random.choice(n_exams, size=n_exams, replace=True)
    yt_b  = all_y_true[idx]
    yp_b  = all_y_pred[idx]
    ypr_b = all_y_proba[idx]

    if len(np.unique(yt_b)) < 2:
        continue

    m = compute_metrics(yt_b, yp_b, ypr_b)
    for key in ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'Accuracy']:
        val = m[key]
        if not np.isnan(val):
            boot_metrics[key].append(val)

print(f'Bootstrap 95% CIs (N={N_BOOTSTRAP} resamples, exam level):')
print('-' * 60)

boot_rows = []
for metric in ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'Accuracy']:
    vals    = np.array(boot_metrics[metric])
    lo, hi  = np.percentile(vals, [2.5, 97.5])
    point   = exam_metrics[metric]
    n_valid = len(vals)
    boot_rows.append({
        'metric':             metric,
        'point_estimate':     point,
        'ci_lower_2_5':       lo,
        'ci_upper_97_5':      hi,
        'n_bootstrap_valid':  n_valid,
    })
    print(f'  {metric:<14}: {point:.4f}  95% CI [{lo:.4f}, {hi:.4f}]  (n_valid={n_valid})')

boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv(os.path.join(OUTPUT_DIR, 'bootstrap_cis.csv'), index=False)
print(f'\nSaved: bootstrap_cis.csv')

## Feature Importances — XGBoost

Aggregate XGBoost feature importances (F-score / gain) across all LOGO folds.
Each feature's mean importance is computed over the folds in which it was selected.

Note: feature names are `feat_0` … `feat_255` (indices into the 256-dim EfficientNetB0 embedding).
These represent learned convolutional activations, not radiomics features.

In [ ]:
feat_imp_rows = []
for feat, imps in feature_importance_accum.items():
    feat_imp_rows.append({
        'feature':            feat,
        'mean_importance':    float(np.mean(imps)),
        'std_importance':     float(np.std(imps)),
        'n_folds_selected':   len(imps),
    })

feat_imp_df = (
    pd.DataFrame(feat_imp_rows)
    .sort_values('mean_importance', ascending=False)
    .reset_index(drop=True)
)

print(f'Top 20 features by mean importance across LOGO folds (N unique={len(feat_imp_df)}):')
print(feat_imp_df.head(20).to_string(index=False))

feat_out = os.path.join(OUTPUT_DIR, 'feature_importances.csv')
feat_imp_df.to_csv(feat_out, index=False)
print(f'\nSaved: {feat_out}')

print('\n' + '=' * 60)
print('All outputs saved to:', OUTPUT_DIR)
print('  - fold_*/predictions.csv  (per-fold exam predictions)')
print('  - summary_metrics_exam_level.csv')
print('  - summary_metrics_animal_level.csv')
print('  - animal_level_metrics.csv')
print('  - animal_level_weighted_vote.csv')
print('  - temporal_analysis.csv')
print('  - bootstrap_cis.csv')
print('  - feature_importances.csv')